## Data Loading

In [4]:
%%time

import kpex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = r"..\input_data\results-filtered-dataset.txt"

feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms



CPU times: total: 0 ns
Wall time: 0 ns


In [5]:
# Loading and preprocessing data
cat_df = kpex_dev.load_excel_data(excel_path, timeout_replacement)
kpex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kpex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kpex_dev.assign_weights(cat_df, scheme, epsilon)

# kpex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kpex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,...,UMKP-S,UMKP-R,UMKP-P,UMKP+-S,UMKP+-R,UMKP+-P,kPEX,kPEX-S,kPEX-R,kPEX-P
0,12month1,20,52328,1585220,30.294,9214,57,19,39,16785.102564,...,1800.000000,1800.000000,1800.000000,131.0240,121.2490,117.7020,116.0158,124.7390,125.8084,126.1804
1,130bit,8,584,6059,10.375,322,12,8,5,564.400000,...,3.196156,1.179364,0.653300,2.3643,0.7011,0.8350,1.4934,0.4922,0.9836,1.7554
2,130bit,9,584,6059,10.375,322,12,8,5,564.400000,...,39.330518,2.159862,2.485482,34.7253,3.2734,3.0159,38.6679,4.9357,22.5385,47.5120
3,130bit,10,583,6051,10.379,321,12,9,4,559.500000,...,9.695851,7.271823,1.946958,9.4931,3.0985,2.4184,10.8117,2.0049,10.4852,12.4310
4,144,4,144319,1072349,7.430,26,9,7,3,140266.333333,...,0.588537,0.539013,0.755349,1.0679,1.0694,0.9626,0.9761,1.0700,1.0626,1.1065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4351,zhishi-zhwiki-internallink,7,4666,248342,53.224,2014,99,19,81,2293.913580,...,2.349277,2.272406,2.238443,1.8562,2.2227,2.2188,1.8133,1.8370,2.2205,2.1942
4352,zhishi-zhwiki-internallink,8,6109,313420,51.305,2421,101,19,83,2731.265060,...,3.851166,3.097895,2.829056,2.8021,2.4912,2.5297,2.8437,2.3888,2.3317,2.8890
4353,zhishi-zhwiki-internallink,9,5981,309664,51.775,2406,101,20,82,2688.695122,...,6.984801,2.970734,2.855774,2.4142,2.4424,2.7942,2.4207,2.5012,2.5222,2.9436
4354,zhishi-zhwiki-internallink,10,4419,244006,55.217,1989,99,22,78,2204.871795,...,13.381666,7.638818,3.442978,1.8692,1.8513,1.6981,1.7289,1.7092,1.7741,1.7088


# Training with  XGBoost (Extreme Gradient Boosting)



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [6]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(
    cat_df,
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties
)
# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Expanded 7 rows with near ties using tolerance 0.0%.
Label ↔ Algorithm mapping:
  0 ➔ UMKP+-P
  1 ➔ UMKP+-R
  2 ➔ UMKP+-S
  3 ➔ UMKP-P
  4 ➔ UMKP-R
  5 ➔ UMKP-S
  6 ➔ kPEX
  7 ➔ kPEX-P
  8 ➔ kPEX-R
  9 ➔ kPEX-S


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report


# Train XGBoost with sample weights
xgb_strat_y = XGBClassifier(
     objective="multi:softprob",
     num_class=len(algorithms),      # = 10 in your case
     n_estimators=400,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     subsample=0.8,     # means that XGBoost would randomly sample portion of training data prior to growing trees, prevents overfitting.
     use_label_encoder=False,
     random_state=42
 )

xgb_strat_y.fit(X_train, y_train, sample_weight=weights_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_class=10, ...)

Feature Importances

In [8]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_strat_y.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
10  shell_minSize    0.139441
7       shell_num    0.105879
2               m    0.100965
1               n    0.100475
8   shell_avgSize    0.100189
5        core_max    0.094664
3           d_avg    0.086485
4           d_max    0.083430
6        core_min    0.075299
0         k_value    0.064034
11  shell_stdSize    0.049140
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [9]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=xgb_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 48.91%

Precision by Algorithm:
  UMKP-S: 49.35% (154 predictions)
  UMKP-R: 48.42% (95 predictions)
  UMKP-P: 65.52% (232 predictions)
  UMKP+-S: 48.21% (56 predictions)
  UMKP+-R: 25.00% (24 predictions)
  UMKP+-P: 32.14% (56 predictions)
  kPEX: 49.28% (138 predictions)
  kPEX-S: 37.74% (53 predictions)
  kPEX-R: 20.00% (15 predictions)
  kPEX-P: 22.00% (50 predictions)

Recall by Algorithm:
  UMKP-S: 53.15% (76/143 cases)
  UMKP-R: 44.23% (46/104 cases)
  UMKP-P: 68.78% (152/221 cases)
  UMKP+-S: 39.71% (27/68 cases)
  UMKP+-R: 18.75% (6/32 cases)
  UMKP+-P: 33.93% (19/56 cases)
  kPEX: 56.45% (70/124 cases)
  kPEX-S: 40.00% (20/50 cases)
  kPEX-R: 12.00% (3/25 cases)
  kPEX-P: 20.75% (11/53 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 59256.40
  Total Optimal Time   = 21823.43
  Time Ratio (pred/opt)= 2.715
  Avg Overhead/case ((pred - opt) / opt)    = 134803

#### Re-evalute on entire dataset

In [10]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 85.33%

Precision by Algorithm:
  UMKP-S: 84.38% (736 predictions)
  UMKP-R: 86.00% (507 predictions)
  UMKP-P: 89.86% (1114 predictions)
  UMKP+-S: 87.12% (326 predictions)
  UMKP+-R: 85.42% (144 predictions)
  UMKP+-P: 79.78% (277 predictions)
  kPEX: 83.36% (637 predictions)
  kPEX-S: 82.75% (255 predictions)
  kPEX-R: 83.78% (111 predictions)
  kPEX-P: 78.71% (249 predictions)

Recall by Algorithm:
  UMKP-S: 86.73% (621/716 cases)
  UMKP-R: 83.85% (436/520 cases)
  UMKP-P: 90.67% (1001/1104 cases)
  UMKP+-S: 84.57% (285/337 cases)
  UMKP+-R: 77.85% (123/158 cases)
  UMKP+-P: 80.22% (223/278 cases)
  kPEX: 87.25% (534/612 cases)
  kPEX-S: 84.06% (211/251 cases)
  kPEX-R: 74.40% (93/125 cases)
  kPEX-P: 74.81% (196/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 160241.31
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.442
  Avg Overhead/case ((pr

### **Balanced train-test split based on both Y and k**

In [11]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(
    cat_df,
    test_size = 0.2,
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Expanded 7 rows with near ties using tolerance 0.0%.
Label ↔ Algorithm mapping:
  0 ➔ UMKP+-P
  1 ➔ UMKP+-R
  2 ➔ UMKP+-S
  3 ➔ UMKP-P
  4 ➔ UMKP-R
  5 ➔ UMKP-S
  6 ➔ kPEX
  7 ➔ kPEX-P
  8 ➔ kPEX-R
  9 ➔ kPEX-S


In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report


# Train XGBoost with sample weights
xgb_strat_y_K = XGBClassifier(
     objective="multi:softprob",
     num_class=len(algorithms),      # = 5 in our case
     n_estimators=400,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     subsample=0.8,
     random_state=42
 )

xgb_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_class=10, ...)

Feature Importances

In [13]:
# --- Feature Importance ---
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
10  shell_minSize    0.111716
5        core_max    0.108768
2               m    0.102499
3           d_avg    0.101983
1               n    0.101633
8   shell_avgSize    0.100448
7       shell_num    0.091953
4           d_max    0.081902
6        core_min    0.077686
0         k_value    0.073463
11  shell_stdSize    0.047949
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [14]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=xgb_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.17%

Precision by Algorithm:
  UMKP-S: 54.42% (147 predictions)
  UMKP-R: 45.63% (103 predictions)
  UMKP-P: 62.70% (244 predictions)
  UMKP+-S: 40.26% (77 predictions)
  UMKP+-R: 52.17% (23 predictions)
  UMKP+-P: 26.53% (49 predictions)
  kPEX: 51.88% (133 predictions)
  kPEX-S: 41.18% (51 predictions)
  kPEX-R: 18.75% (16 predictions)
  kPEX-P: 30.00% (30 predictions)

Recall by Algorithm:
  UMKP-S: 55.94% (80/143 cases)
  UMKP-R: 44.76% (47/105 cases)
  UMKP-P: 68.92% (153/222 cases)
  UMKP+-S: 45.59% (31/68 cases)
  UMKP+-R: 35.29% (12/34 cases)
  UMKP+-P: 25.93% (14/54 cases)
  kPEX: 55.65% (69/124 cases)
  kPEX-S: 44.00% (22/50 cases)
  kPEX-R: 12.50% (3/24 cases)
  kPEX-P: 17.31% (9/52 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 47241.58
  Total Optimal Time   = 17300.53
  Time Ratio (pred/opt)= 2.731
  Avg Overhead/case ((pred - opt) / opt)    = 364.8

#### Re-evalute on entire dataset

In [15]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 85.31%

Precision by Algorithm:
  UMKP-S: 86.19% (717 predictions)
  UMKP-R: 85.08% (516 predictions)
  UMKP-P: 88.30% (1137 predictions)
  UMKP+-S: 83.48% (339 predictions)
  UMKP+-R: 86.81% (144 predictions)
  UMKP+-P: 79.85% (273 predictions)
  kPEX: 84.19% (626 predictions)
  kPEX-S: 83.40% (253 predictions)
  kPEX-R: 80.70% (114 predictions)
  kPEX-P: 83.97% (237 predictions)

Recall by Algorithm:
  UMKP-S: 86.31% (618/716 cases)
  UMKP-R: 84.42% (439/520 cases)
  UMKP-P: 90.94% (1004/1104 cases)
  UMKP+-S: 84.27% (284/337 cases)
  UMKP+-R: 79.11% (125/158 cases)
  UMKP+-P: 78.78% (219/278 cases)
  kPEX: 86.44% (529/612 cases)
  kPEX-S: 84.46% (212/251 cases)
  kPEX-R: 74.40% (93/125 cases)
  kPEX-P: 75.95% (199/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 151820.08
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.366
  Avg Overhead/case ((pr

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [16]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,...,UMKP-S,UMKP-R,UMKP-P,UMKP+-S,UMKP+-R,UMKP+-P,kPEX,kPEX-S,kPEX-R,kPEX-P


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [17]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [18]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (3485, 25)
Test set shape:  (871, 25)


In [19]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ UMKP+-P
  1 ➔ UMKP+-R
  2 ➔ UMKP+-S
  3 ➔ UMKP-P
  4 ➔ UMKP-R
  5 ➔ UMKP-S
  6 ➔ kPEX
  7 ➔ kPEX-P
  8 ➔ kPEX-R
  9 ➔ kPEX-S


In [20]:
from xgboost import XGBClassifier

# Train XGBoost with sample weights
xgb_holdout_strat_y = XGBClassifier(
     objective="multi:softprob",
     num_class=len(data_splits.algorithms),      # = 5 in our case
     n_estimators=200,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     use_label_encoder=False,
    #  random_state=42
 )

xgb_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_class=10, ...)

Feature Importances

In [21]:
# --- Feature Importance ---
importances = pd.DataFrame({
    'feature': X_train_.columns,
    'importance': xgb_holdout_strat_y.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
10  shell_minSize    0.138095
2               m    0.134224
5        core_max    0.097432
1               n    0.093881
3           d_avg    0.093082
6        core_min    0.085106
8   shell_avgSize    0.079244
4           d_max    0.075651
7       shell_num    0.070445
11  shell_stdSize    0.069347
0         k_value    0.063493
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [22]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=xgb_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # 
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 43.63%

Precision by Algorithm:
  UMKP-S: 48.85% (131 predictions)
  UMKP-R: 48.51% (101 predictions)
  UMKP-P: 55.65% (248 predictions)
  UMKP+-S: 33.33% (54 predictions)
  UMKP+-R: 29.41% (34 predictions)
  UMKP+-P: 20.93% (43 predictions)
  kPEX: 39.74% (151 predictions)
  kPEX-S: 47.27% (55 predictions)
  kPEX-R: 10.53% (19 predictions)
  kPEX-P: 11.43% (35 predictions)

Recall by Algorithm:
  UMKP-S: 44.76% (64/143 cases)
  UMKP-R: 47.12% (49/104 cases)
  UMKP-P: 62.44% (138/221 cases)
  UMKP+-S: 26.87% (18/67 cases)
  UMKP+-R: 31.25% (10/32 cases)
  UMKP+-P: 16.07% (9/56 cases)
  kPEX: 48.78% (60/123 cases)
  kPEX-S: 52.00% (26/50 cases)
  kPEX-R: 8.00% (2/25 cases)
  kPEX-P: 7.69% (4/52 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 69586.57
  Total Optimal Time   = 21423.42
  Time Ratio (pred/opt)= 3.248
  Avg Overhead/case ((pred - opt) / opt)    = 388267.7

#### Re-evalute on entire dataset

In [23]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 81.84%

Precision by Algorithm:
  UMKP-S: 82.93% (709 predictions)
  UMKP-R: 82.58% (511 predictions)
  UMKP-P: 85.50% (1131 predictions)
  UMKP+-S: 84.19% (310 predictions)
  UMKP+-R: 79.08% (153 predictions)
  UMKP+-P: 78.60% (257 predictions)
  kPEX: 76.46% (684 predictions)
  kPEX-S: 83.47% (248 predictions)
  kPEX-R: 74.80% (123 predictions)
  kPEX-P: 79.13% (230 predictions)

Recall by Algorithm:
  UMKP-S: 82.12% (588/716 cases)
  UMKP-R: 81.15% (422/520 cases)
  UMKP-P: 87.59% (967/1104 cases)
  UMKP+-S: 77.74% (262/337 cases)
  UMKP+-R: 76.58% (121/158 cases)
  UMKP+-P: 73.02% (203/278 cases)
  kPEX: 85.78% (525/612 cases)
  kPEX-S: 82.47% (207/251 cases)
  kPEX-R: 73.60% (92/125 cases)
  kPEX-P: 69.47% (182/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 171149.85
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.540
  Avg Overhead/case ((pre

### **Balanced train-test split based on Y and K**

In [24]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (3484, 25)
Test set shape:  (872, 25)


In [25]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.0  # algorithms within tie_tolerance% of min treated as ties

)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ UMKP+-P
  1 ➔ UMKP+-R
  2 ➔ UMKP+-S
  3 ➔ UMKP-P
  4 ➔ UMKP-R
  5 ➔ UMKP-S
  6 ➔ kPEX
  7 ➔ kPEX-P
  8 ➔ kPEX-R
  9 ➔ kPEX-S


In [26]:
from xgboost import XGBClassifier

# Train XGBoost with sample weights
xgb_holdout_strat_y_K = XGBClassifier(
     objective="multi:softprob",
     num_class=len(data_splits.algorithms),      # = 5 in our case
     n_estimators=150,
     max_depth=15,
     learning_rate=0.75,              # you can tune this later
     use_label_encoder=False,
     random_state=42
 )

xgb_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.75, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=15,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_class=10, ...)

Feature Importances

In [27]:
# --- Feature Importance ---
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_holdout_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
1               n    0.144650
2               m    0.128074
5        core_max    0.126990
10  shell_minSize    0.106930
3           d_avg    0.104755
0         k_value    0.073016
6        core_min    0.066322
7       shell_num    0.064081
8   shell_avgSize    0.063734
11  shell_stdSize    0.061019
4           d_max    0.060429
9   shell_maxSize    0.000000


#### Inference on 20% dataset

In [28]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=xgb_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 44.27%

Precision by Algorithm:
  UMKP-S: 44.14% (145 predictions)
  UMKP-R: 41.58% (101 predictions)
  UMKP-P: 58.65% (237 predictions)
  UMKP+-S: 33.90% (59 predictions)
  UMKP+-R: 35.29% (17 predictions)
  UMKP+-P: 23.40% (47 predictions)
  kPEX: 47.26% (146 predictions)
  kPEX-S: 38.89% (54 predictions)
  kPEX-R: 19.05% (21 predictions)
  kPEX-P: 22.22% (45 predictions)

Recall by Algorithm:
  UMKP-S: 44.76% (64/143 cases)
  UMKP-R: 40.38% (42/104 cases)
  UMKP-P: 62.61% (139/222 cases)
  UMKP+-S: 29.85% (20/67 cases)
  UMKP+-R: 18.75% (6/32 cases)
  UMKP+-P: 19.64% (11/56 cases)
  kPEX: 56.10% (69/123 cases)
  kPEX-S: 42.00% (21/50 cases)
  kPEX-R: 16.00% (4/25 cases)
  kPEX-P: 19.23% (10/52 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 67947.27
  Total Optimal Time   = 20051.12
  Time Ratio (pred/opt)= 3.389
  Avg Overhead/case ((pred - opt) / opt)    = 20825

#### Re-evalute on entire dataset

In [29]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kpex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=xgb_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='../outputs/XGB_results/tie_tol/0/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kpex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "../outputs/XGB_results/tie_tol/0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 84.80%

Precision by Algorithm:
  UMKP-S: 84.07% (728 predictions)
  UMKP-R: 85.80% (507 predictions)
  UMKP-P: 88.65% (1119 predictions)
  UMKP+-S: 84.88% (324 predictions)
  UMKP+-R: 87.05% (139 predictions)
  UMKP+-P: 79.85% (268 predictions)
  kPEX: 83.70% (638 predictions)
  kPEX-S: 82.10% (257 predictions)
  kPEX-R: 77.69% (121 predictions)
  kPEX-P: 80.78% (255 predictions)

Recall by Algorithm:
  UMKP-S: 85.47% (612/716 cases)
  UMKP-R: 83.65% (435/520 cases)
  UMKP-P: 89.86% (992/1104 cases)
  UMKP+-S: 81.90% (276/337 cases)
  UMKP+-R: 76.58% (121/158 cases)
  UMKP+-P: 77.70% (216/278 cases)
  kPEX: 87.42% (535/612 cases)
  kPEX-S: 84.06% (211/251 cases)
  kPEX-R: 76.00% (95/125 cases)
  kPEX-P: 78.63% (206/262 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 166076.44
  Total Optimal Time   = 111146.05
  Time Ratio (pred/opt)= 1.494
  Avg Overhead/case ((pre